In [0]:
DESCRIBE TABLE pondolens.dev_bronze.unemployment_rate;

SELECT
  TRIM(region) AS source_region,
  TRIM(geographic_name) AS geographic_name,
  TRIM(geographic_level) AS geographic_level,
  MIN(TRY_CAST(TRIM(year) AS INT)) AS valid_from_year,
  MAX(TRY_CAST(TRIM(year) AS INT)) AS valid_to_year,
  COUNT(*) AS row_count
FROM pondolens.dev_bronze.unemployment_rate
GROUP BY
  TRIM(region),
  TRIM(geographic_name),
  TRIM(geographic_level)
ORDER BY
  geographic_level,
  source_region,
  geographic_name;


  SELECT
  year AS raw_year,
  COUNT(*) AS row_count
FROM pondolens.dev_bronze.unemployment_rate
GROUP BY year
ORDER BY year;

SELECT CAST(CAST(year AS DOUBLE) AS INT) FROM pondolens.dev_bronze.unemployment_rate;



WITH parsed AS (
  SELECT
    TRIM(region) AS source_region,
    LOWER(TRIM(geographic_level)) AS source_geographic_level,
    CAST(CAST(year AS DOUBLE) AS INT) AS year_int
  FROM pondolens.dev_bronze.unemployment_rate
)
SELECT
  source_region,
  source_geographic_level,
  MIN(year_int) AS valid_from_year,
  MAX(year_int) AS valid_to_year,
  COUNT(*) AS row_count
FROM parsed
WHERE year_int BETWEEN 2021 AND 2025
GROUP BY source_region, source_geographic_level
ORDER BY source_geographic_level, source_region;


WITH parsed AS (
  SELECT
    TRIM(region) AS source_region,
    LOWER(TRIM(geographic_level)) AS source_geographic_level,
    CAST(CAST(year AS DOUBLE) AS INT) AS year_int
  FROM pondolens.dev_bronze.unemployment_rate
)
SELECT
  source_region,
  MIN(year_int) AS valid_from_year,
  MAX(year_int) AS valid_to_year,
  COUNT(*) AS row_count
FROM parsed
WHERE source_geographic_level = 'region'
  AND year_int BETWEEN 2021 AND 2025
GROUP BY source_region
ORDER BY source_region;


WITH regional_source AS (
  SELECT
    TRIM(region) AS source_region,
    CAST(CAST(year AS DOUBLE) AS INT) AS year_int
  FROM pondolens.dev_bronze.unemployment_rate
  WHERE LOWER(TRIM(geographic_level)) = 'region'
    AND CAST(CAST(year AS DOUBLE) AS INT) BETWEEN 2021 AND 2025
),
aggregated AS (
  SELECT
    source_region,
    MIN(year_int) AS valid_from_year,
    MAX(year_int) AS valid_to_year,
    COUNT(*) AS row_count
  FROM regional_source
  GROUP BY source_region
)
SELECT
  source_region,

  CASE
    WHEN source_region LIKE '%(BARMM/ARMM)%' THEN 'BARMM'
    WHEN source_region LIKE '%(BARMM)%' THEN 'BARMM'
    WHEN source_region LIKE '%(CAR)%' THEN 'CAR'
    WHEN source_region LIKE '%(NCR)%' THEN 'NCR'
    WHEN source_region LIKE '%(NIR)%' THEN 'NIR'
    WHEN source_region LIKE '%(CALABARZON)%' THEN 'IVA'
    WHEN source_region = 'REGION IVA CALABARZON' THEN 'IVA'
    WHEN source_region LIKE '%(ILOCOS REGION)%' THEN 'I'
    WHEN source_region LIKE '%(CAGAYAN VALLEY)%' THEN 'II'
    WHEN source_region LIKE '%(CENTRAL LUZON)%' THEN 'III'
    WHEN source_region LIKE '%(ZAMBOANGA PENINSULA)%' THEN 'IX'
    WHEN source_region LIKE '%(BICOL REGION)%' THEN 'V'
    WHEN source_region LIKE '%(WESTERN VISAYAS)%' THEN 'VI'
    WHEN source_region LIKE '%(CENTRAL VISAYAS)%' THEN 'VII'
    WHEN source_region LIKE '%(EASTERN VISAYAS)%' THEN 'VIII'
    WHEN source_region LIKE '%(NORTHERN MINDANAO)%' THEN 'X'
    WHEN source_region LIKE '%(DAVAO REGION)%' THEN 'XI'
    WHEN source_region LIKE '%(SOCCSKSARGEN)%' THEN 'XII'
    WHEN source_region LIKE '%(CARAGA)%' THEN 'XIII'
    WHEN source_region = 'MIMAROPA REGION' THEN 'IVB'
    ELSE NULL
  END AS source_code,

  CASE
    WHEN source_region LIKE '%(BARMM/ARMM)%'
      OR source_region LIKE '%(BARMM)%' THEN 'BARMM'
    WHEN source_region LIKE '%(CAR)%' THEN 'CAR'
    WHEN source_region LIKE '%(NCR)%' THEN 'NCR'
    WHEN source_region LIKE '%(NIR)%' THEN 'NIR'
    WHEN source_region LIKE '%(CALABARZON)%'
      OR source_region = 'REGION IVA CALABARZON' THEN 'IVA'
    WHEN source_region LIKE '%(ILOCOS REGION)%' THEN 'I'
    WHEN source_region LIKE '%(CAGAYAN VALLEY)%' THEN 'II'
    WHEN source_region LIKE '%(CENTRAL LUZON)%' THEN 'III'
    WHEN source_region LIKE '%(ZAMBOANGA PENINSULA)%' THEN 'IX'
    WHEN source_region LIKE '%(BICOL REGION)%' THEN 'V'
    WHEN source_region LIKE '%(WESTERN VISAYAS)%' THEN 'VI'
    WHEN source_region LIKE '%(CENTRAL VISAYAS)%' THEN 'VII'
    WHEN source_region LIKE '%(EASTERN VISAYAS)%' THEN 'VIII'
    WHEN source_region LIKE '%(NORTHERN MINDANAO)%' THEN 'X'
    WHEN source_region LIKE '%(DAVAO REGION)%' THEN 'XI'
    WHEN source_region LIKE '%(SOCCSKSARGEN)%' THEN 'XII'
    WHEN source_region LIKE '%(CARAGA)%' THEN 'XIII'
    WHEN source_region = 'MIMAROPA REGION' THEN 'IVB'
    ELSE NULL
  END AS region_code,

  CASE
    WHEN source_region LIKE '%(BARMM/ARMM)%'
      OR source_region LIKE '%(BARMM)%'
      THEN 'BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO'
    WHEN source_region LIKE '%(CAR)%'
      THEN 'CORDILLERA ADMINISTRATIVE REGION'
    WHEN source_region LIKE '%(NCR)%'
      THEN 'NATIONAL CAPITAL REGION'
    WHEN source_region LIKE '%(NIR)%'
      THEN 'NEGROS ISLAND REGION'
    WHEN source_region LIKE '%(CALABARZON)%'
      OR source_region = 'REGION IVA CALABARZON'
      THEN 'CALABARZON'
    WHEN source_region LIKE '%(ILOCOS REGION)%'
      THEN 'ILOCOS REGION'
    WHEN source_region LIKE '%(CAGAYAN VALLEY)%'
      THEN 'CAGAYAN VALLEY'
    WHEN source_region LIKE '%(CENTRAL LUZON)%'
      THEN 'CENTRAL LUZON'
    WHEN source_region LIKE '%(ZAMBOANGA PENINSULA)%'
      THEN 'ZAMBOANGA PENINSULA'
    WHEN source_region LIKE '%(BICOL REGION)%'
      THEN 'BICOL REGION'
    WHEN source_region LIKE '%(WESTERN VISAYAS)%'
      THEN 'WESTERN VISAYAS'
    WHEN source_region LIKE '%(CENTRAL VISAYAS)%'
      THEN 'CENTRAL VISAYAS'
    WHEN source_region LIKE '%(EASTERN VISAYAS)%'
      THEN 'EASTERN VISAYAS'
    WHEN source_region LIKE '%(NORTHERN MINDANAO)%'
      THEN 'NORTHERN MINDANAO'
    WHEN source_region LIKE '%(DAVAO REGION)%'
      THEN 'DAVAO REGION'
    WHEN source_region LIKE '%(SOCCSKSARGEN)%'
      THEN 'SOCCSKSARGEN'
    WHEN source_region LIKE '%(CARAGA)%'
      THEN 'CARAGA'
    WHEN source_region = 'MIMAROPA REGION'
      THEN 'MIMAROPA REGION'
    ELSE NULL
  END AS region,

  'region' AS geographic_level,
  valid_from_year,
  valid_to_year,
  row_count

FROM aggregated
ORDER BY region_code, source_region;
------------------------------------------------------------------------------------------------

CREATE TABLE IF NOT EXISTS pondolens.dev_silver.shared_region_crosswalk (
  source_region STRING,
  source_code STRING,
  region_code STRING,
  region STRING,
  geographic_level STRING,
  valid_from_year INT,
  valid_to_year INT
);


INSERT INTO pondolens.dev_silver.shared_region_crosswalk (source_region, source_code, region_code, region_name, geographic_level, valid_from_year, valid_to_year)
SELECT
  source_region,
  source_code,
  region_code,
  region,
  geographic_level,
  valid_from_year,
  valid_to_year
FROM (
  WITH regional_source AS (
    SELECT
      TRIM(region) AS source_region,
      CAST(CAST(year AS DOUBLE) AS INT) AS year_int
    FROM pondolens.dev_bronze.unemployment_rate
    WHERE LOWER(TRIM(geographic_level)) = 'region'
      AND CAST(CAST(year AS DOUBLE) AS INT) BETWEEN 2021 AND 2025
  ),
  aggregated AS (
    SELECT
      source_region,
      MIN(year_int) AS valid_from_year,
      MAX(year_int) AS valid_to_year
    FROM regional_source
    GROUP BY source_region
  )
  SELECT
    source_region,

    CASE
      WHEN source_region LIKE '%(BARMM/ARMM)%' THEN 'BARMM/ARMM'
      WHEN source_region LIKE '%(BARMM)%' THEN 'BARMM'
      WHEN source_region LIKE '%(CAR)%' THEN 'CAR'
      WHEN source_region LIKE '%(NCR)%' THEN 'NCR'
      WHEN source_region LIKE '%(NIR)%' THEN 'NIR'
      WHEN source_region LIKE '%(CALABARZON)%'
        OR source_region = 'REGION IVA CALABARZON' THEN 'IVA'
      WHEN source_region LIKE '%(ILOCOS REGION)%' THEN 'I'
      WHEN source_region LIKE '%(CAGAYAN VALLEY)%' THEN 'II'
      WHEN source_region LIKE '%(CENTRAL LUZON)%' THEN 'III'
      WHEN source_region LIKE '%(ZAMBOANGA PENINSULA)%' THEN 'IX'
      WHEN source_region LIKE '%(BICOL REGION)%' THEN 'V'
      WHEN source_region LIKE '%(WESTERN VISAYAS)%' THEN 'VI'
      WHEN source_region LIKE '%(CENTRAL VISAYAS)%' THEN 'VII'
      WHEN source_region LIKE '%(EASTERN VISAYAS)%' THEN 'VIII'
      WHEN source_region LIKE '%(NORTHERN MINDANAO)%' THEN 'X'
      WHEN source_region LIKE '%(DAVAO REGION)%' THEN 'XI'
      WHEN source_region LIKE '%(SOCCSKSARGEN)%' THEN 'XII'
      WHEN source_region LIKE '%(CARAGA)%' THEN 'XIII'
      WHEN source_region = 'MIMAROPA REGION' THEN 'IVB'
      ELSE NULL
    END AS source_code,

    CASE
      WHEN source_region LIKE '%(BARMM/ARMM)%'
        OR source_region LIKE '%(BARMM)%' THEN 'BARMM'
      WHEN source_region LIKE '%(CAR)%' THEN 'CAR'
      WHEN source_region LIKE '%(NCR)%' THEN 'NCR'
      WHEN source_region LIKE '%(NIR)%' THEN 'NIR'
      WHEN source_region LIKE '%(CALABARZON)%'
        OR source_region = 'REGION IVA CALABARZON' THEN 'IVA'
      WHEN source_region LIKE '%(ILOCOS REGION)%' THEN 'I'
      WHEN source_region LIKE '%(CAGAYAN VALLEY)%' THEN 'II'
      WHEN source_region LIKE '%(CENTRAL LUZON)%' THEN 'III'
      WHEN source_region LIKE '%(ZAMBOANGA PENINSULA)%' THEN 'IX'
      WHEN source_region LIKE '%(BICOL REGION)%' THEN 'V'
      WHEN source_region LIKE '%(WESTERN VISAYAS)%' THEN 'VI'
      WHEN source_region LIKE '%(CENTRAL VISAYAS)%' THEN 'VII'
      WHEN source_region LIKE '%(EASTERN VISAYAS)%' THEN 'VIII'
      WHEN source_region LIKE '%(NORTHERN MINDANAO)%' THEN 'X'
      WHEN source_region LIKE '%(DAVAO REGION)%' THEN 'XI'
      WHEN source_region LIKE '%(SOCCSKSARGEN)%' THEN 'XII'
      WHEN source_region LIKE '%(CARAGA)%' THEN 'XIII'
      WHEN source_region = 'MIMAROPA REGION' THEN 'IVB'
      ELSE NULL
    END AS region_code,

    CASE
      WHEN source_region LIKE '%(BARMM/ARMM)%'
        OR source_region LIKE '%(BARMM)%'
        THEN 'BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO'
      WHEN source_region LIKE '%(CAR)%'
        THEN 'CORDILLERA ADMINISTRATIVE REGION'
      WHEN source_region LIKE '%(NCR)%'
        THEN 'NATIONAL CAPITAL REGION'
      WHEN source_region LIKE '%(NIR)%'
        THEN 'NEGROS ISLAND REGION'
      WHEN source_region LIKE '%(CALABARZON)%'
        OR source_region = 'REGION IVA CALABARZON'
        THEN 'CALABARZON'
      WHEN source_region LIKE '%(ILOCOS REGION)%'
        THEN 'ILOCOS REGION'
      WHEN source_region LIKE '%(CAGAYAN VALLEY)%'
        THEN 'CAGAYAN VALLEY'
      WHEN source_region LIKE '%(CENTRAL LUZON)%'
        THEN 'CENTRAL LUZON'
      WHEN source_region LIKE '%(ZAMBOANGA PENINSULA)%'
        THEN 'ZAMBOANGA PENINSULA'
      WHEN source_region LIKE '%(BICOL REGION)%'
        THEN 'BICOL REGION'
      WHEN source_region LIKE '%(WESTERN VISAYAS)%'
        THEN 'WESTERN VISAYAS'
      WHEN source_region LIKE '%(CENTRAL VISAYAS)%'
        THEN 'CENTRAL VISAYAS'
      WHEN source_region LIKE '%(EASTERN VISAYAS)%'
        THEN 'EASTERN VISAYAS'
      WHEN source_region LIKE '%(NORTHERN MINDANAO)%'
        THEN 'NORTHERN MINDANAO'
      WHEN source_region LIKE '%(DAVAO REGION)%'
        THEN 'DAVAO REGION'
      WHEN source_region LIKE '%(SOCCSKSARGEN)%'
        THEN 'SOCCSKSARGEN'
      WHEN source_region LIKE '%(CARAGA)%'
        THEN 'CARAGA'
      WHEN source_region = 'MIMAROPA REGION'
        THEN 'MIMAROPA REGION'
      ELSE NULL
    END AS region,

    'region' AS geographic_level,
    valid_from_year,
    valid_to_year
  FROM aggregated
) AS new_crosswalk
WHERE NOT EXISTS (
  SELECT 1
  FROM pondolens.dev_silver.shared_region_crosswalk
);

DESCRIBE pondolens.dev_silver.shared_region_crosswalk;

MERGE INTO pondolens.dev_silver.shared_region_crosswalk AS target
USING (
  WITH raw_years AS (
    SELECT
      TRIM(region) AS source_region,
      MIN(CAST(CAST(year AS DOUBLE) AS INT)) AS valid_from_year,
      MAX(CAST(CAST(year AS DOUBLE) AS INT)) AS valid_to_year
    FROM pondolens.dev_bronze.unemployment_rate
    WHERE LOWER(TRIM(geographic_level)) = 'region'
      AND CAST(CAST(year AS DOUBLE) AS INT) BETWEEN 2021 AND 2025
    GROUP BY TRIM(region)
  ),
  region_map AS (
    SELECT * FROM VALUES
      ('BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO (BARMM)', 'BARMM', 'BARMM', 'BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO'),
      ('BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO (BARMM/ARMM)', 'BARMM/ARMM', 'BARMM', 'BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO'),
      ('CORDILLERA ADMINISTRATIVE REGION (CAR)', 'CAR', 'CAR', 'CORDILLERA ADMINISTRATIVE REGION'),
      ('REGION I (ILOCOS REGION)', 'I', 'I', 'ILOCOS REGION'),
      ('REGION II (CAGAYAN VALLEY)', 'II', 'II', 'CAGAYAN VALLEY'),
      ('REGION III (CENTRAL LUZON)', 'III', 'III', 'CENTRAL LUZON'),
      ('REGION IV-A (CALABARZON)', 'IV-A', 'IVA', 'CALABARZON'),
      ('REGION IVA (CALABARZON)', 'IVA', 'IVA', 'CALABARZON'),
      ('REGION IVA CALABARZON', 'IVA', 'IVA', 'CALABARZON'),
      ('MIMAROPA REGION', NULL, 'IVB', 'MIMAROPA REGION'),
      ('REGION IX (ZAMBOANGA PENINSULA)', 'IX', 'IX', 'ZAMBOANGA PENINSULA'),
      ('NATIONAL CAPITAL REGION (NCR)', 'NCR', 'NCR', 'NATIONAL CAPITAL REGION'),
      ('NEGROS ISLAND REGION (NIR)', 'NIR', 'NIR', 'NEGROS ISLAND REGION'),
      ('REGION V (BICOL REGION)', 'V', 'V', 'BICOL REGION'),
      ('REGION VI (WESTERN VISAYAS)', 'VI', 'VI', 'WESTERN VISAYAS'),
      ('REGION VII (CENTRAL VISAYAS)', 'VII', 'VII', 'CENTRAL VISAYAS'),
      ('REGION VIII (EASTERN VISAYAS)', 'VIII', 'VIII', 'EASTERN VISAYAS'),
      ('REGION X (NORTHERN MINDANAO)', 'X', 'X', 'NORTHERN MINDANAO'),
      ('REGION XI (DAVAO REGION)', 'XI', 'XI', 'DAVAO REGION'),
      ('REGION XII (SOCCSKSARGEN)', 'XII', 'XII', 'SOCCSKSARGEN'),
      ('REGION XIII (CARAGA)', 'XIII', 'XIII', 'CARAGA')
    AS m(source_region, source_code, region_code, region_name)
  )
  SELECT
    'Philippines' AS country,
    m.source_region,
    m.region_code,
    m.region_name,
    y.valid_from_year,
    y.valid_to_year,
    m.source_code,
    'region' AS geographic_level
  FROM region_map m
  INNER JOIN raw_years y
    ON m.source_region = y.source_region
) AS source
ON target.source_region = source.source_region
AND target.geographic_level = source.geographic_level

WHEN MATCHED THEN UPDATE SET
  target.country = source.country,
  target.region_code = source.region_code,
  target.region_name = source.region_name,
  target.valid_from_year = source.valid_from_year,
  target.valid_to_year = source.valid_to_year,
  target.source_code = source.source_code

WHEN NOT MATCHED THEN INSERT (
  country,
  source_region,
  region_code,
  region_name,
  valid_from_year,
  valid_to_year,
  source_code,
  geographic_level
)
VALUES (
  source.country,
  source.source_region,
  source.region_code,
  source.region_name,
  source.valid_from_year,
  source.valid_to_year,
  source.source_code,
  source.geographic_level
);



SELECT
  'Philippines' AS country,
  'test' AS source_region,
  'TEST' AS region_code,
  'Test Region' AS region_name,
  2021 AS valid_from_year,
  2025 AS valid_to_year,
  'TEST' AS source_code,
  'region' AS geographic_level;



  WITH raw_years AS (
  SELECT
    TRIM(region) AS source_region,
    MIN(TRY_CAST(year AS INT)) AS valid_from_year,
    MAX(TRY_CAST(year AS INT)) AS valid_to_year
  FROM pondolens.dev_bronze.unemployment_rate
  WHERE LOWER(TRIM(geographic_level)) = 'region'
    AND TRY_CAST(year AS INT) BETWEEN 2021 AND 2025
  GROUP BY TRIM(region)
),
region_map AS (
  SELECT * FROM VALUES
    ('BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO (BARMM)', 'BARMM', 'BARMM', 'BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO'),
    ('BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO (BARMM/ARMM)', 'BARMM/ARMM', 'BARMM', 'BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO'),
    ('CORDILLERA ADMINISTRATIVE REGION (CAR)', 'CAR', 'CAR', 'CORDILLERA ADMINISTRATIVE REGION'),
    ('REGION I (ILOCOS REGION)', 'I', 'I', 'ILOCOS REGION'),
    ('REGION II (CAGAYAN VALLEY)', 'II', 'II', 'CAGAYAN VALLEY'),
    ('REGION III (CENTRAL LUZON)', 'III', 'III', 'CENTRAL LUZON'),
    ('REGION IV-A (CALABARZON)', 'IV-A', 'IVA', 'CALABARZON'),
    ('REGION IVA (CALABARZON)', 'IVA', 'IVA', 'CALABARZON'),
    ('REGION IVA CALABARZON', 'IVA', 'IVA', 'CALABARZON'),
    ('MIMAROPA REGION', NULL, 'IVB', 'MIMAROPA REGION'),
    ('REGION IX (ZAMBOANGA PENINSULA)', 'IX', 'IX', 'ZAMBOANGA PENINSULA'),
    ('NATIONAL CAPITAL REGION (NCR)', 'NCR', 'NCR', 'NATIONAL CAPITAL REGION'),
    ('NEGROS ISLAND REGION (NIR)', 'NIR', 'NIR', 'NEGROS ISLAND REGION'),
    ('REGION V (BICOL REGION)', 'V', 'V', 'BICOL REGION'),
    ('REGION VI (WESTERN VISAYAS)', 'VI', 'VI', 'WESTERN VISAYAS'),
    ('REGION VII (CENTRAL VISAYAS)', 'VII', 'VII', 'CENTRAL VISAYAS'),
    ('REGION VIII (EASTERN VISAYAS)', 'VIII', 'VIII', 'EASTERN VISAYAS'),
    ('REGION X (NORTHERN MINDANAO)', 'X', 'X', 'NORTHERN MINDANAO'),
    ('REGION XI (DAVAO REGION)', 'XI', 'XI', 'DAVAO REGION'),
    ('REGION XII (SOCCSKSARGEN)', 'XII', 'XII', 'SOCCSKSARGEN'),
    ('REGION XIII (CARAGA)', 'XIII', 'XIII', 'CARAGA')
  AS m(source_region, source_code, region_code, region_name)
)
SELECT
  'Philippines' AS country,
  m.source_region,
  m.region_code,
  m.region_name,
  y.valid_from_year,
  y.valid_to_year,
  m.source_code,
  'region' AS geographic_level
FROM region_map m
JOIN raw_years y
  ON m.source_region = y.source_region;


  SELECT
  region,
  geographic_level,
  year
FROM pondolens.dev_bronze.unemployment_rate
WHERE LOWER(TRIM(geographic_level)) = 'region'
LIMIT 20;


MERGE INTO pondolens.dev_silver.shared_region_crosswalk AS target
USING (
  SELECT
    'Philippines' AS country,
    TRIM(region) AS source_region,

    CASE
      WHEN TRIM(region) LIKE '%(BARMM/ARMM)%' THEN 'BARMM/ARMM'
      WHEN TRIM(region) LIKE '%(BARMM)%' THEN 'BARMM'
      WHEN TRIM(region) LIKE '%(CAR)%' THEN 'CAR'
      WHEN TRIM(region) LIKE '%(NCR)%' THEN 'NCR'
      WHEN TRIM(region) LIKE '%(NIR)%' THEN 'NIR'
      WHEN TRIM(region) LIKE '%(CALABARZON)%' OR TRIM(region) = 'REGION IVA CALABARZON' THEN 'IVA'
      WHEN TRIM(region) LIKE '%(ILOCOS REGION)%' THEN 'I'
      WHEN TRIM(region) LIKE '%(CAGAYAN VALLEY)%' THEN 'II'
      WHEN TRIM(region) LIKE '%(CENTRAL LUZON)%' THEN 'III'
      WHEN TRIM(region) LIKE '%(ZAMBOANGA PENINSULA)%' THEN 'IX'
      WHEN TRIM(region) LIKE '%(BICOL REGION)%' THEN 'V'
      WHEN TRIM(region) LIKE '%(WESTERN VISAYAS)%' THEN 'VI'
      WHEN TRIM(region) LIKE '%(CENTRAL VISAYAS)%' THEN 'VII'
      WHEN TRIM(region) LIKE '%(EASTERN VISAYAS)%' THEN 'VIII'
      WHEN TRIM(region) LIKE '%(NORTHERN MINDANAO)%' THEN 'X'
      WHEN TRIM(region) LIKE '%(DAVAO REGION)%' THEN 'XI'
      WHEN TRIM(region) LIKE '%(SOCCSKSARGEN)%' THEN 'XII'
      WHEN TRIM(region) LIKE '%(CARAGA)%' THEN 'XIII'
      WHEN TRIM(region) = 'MIMAROPA REGION' THEN NULL
    END AS source_code,

    CASE
      WHEN TRIM(region) LIKE '%(BARMM)%' THEN 'BARMM'
      WHEN TRIM(region) LIKE '%(CAR)%' THEN 'CAR'
      WHEN TRIM(region) LIKE '%(NCR)%' THEN 'NCR'
      WHEN TRIM(region) LIKE '%(NIR)%' THEN 'NIR'
      WHEN TRIM(region) LIKE '%(CALABARZON)%' OR TRIM(region) = 'REGION IVA CALABARZON' THEN 'IVA'
      WHEN TRIM(region) LIKE '%(ILOCOS REGION)%' THEN 'I'
      WHEN TRIM(region) LIKE '%(CAGAYAN VALLEY)%' THEN 'II'
      WHEN TRIM(region) LIKE '%(CENTRAL LUZON)%' THEN 'III'
      WHEN TRIM(region) LIKE '%(ZAMBOANGA PENINSULA)%' THEN 'IX'
      WHEN TRIM(region) LIKE '%(BICOL REGION)%' THEN 'V'
      WHEN TRIM(region) LIKE '%(WESTERN VISAYAS)%' THEN 'VI'
      WHEN TRIM(region) LIKE '%(CENTRAL VISAYAS)%' THEN 'VII'
      WHEN TRIM(region) LIKE '%(EASTERN VISAYAS)%' THEN 'VIII'
      WHEN TRIM(region) LIKE '%(NORTHERN MINDANAO)%' THEN 'X'
      WHEN TRIM(region) LIKE '%(DAVAO REGION)%' THEN 'XI'
      WHEN TRIM(region) LIKE '%(SOCCSKSARGEN)%' THEN 'XII'
      WHEN TRIM(region) LIKE '%(CARAGA)%' THEN 'XIII'
      WHEN TRIM(region) = 'MIMAROPA REGION' THEN 'IVB'
    END AS region_code,

    CASE
      WHEN TRIM(region) LIKE '%(BARMM)%' THEN 'BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO'
      WHEN TRIM(region) LIKE '%(CAR)%' THEN 'CORDILLERA ADMINISTRATIVE REGION'
      WHEN TRIM(region) LIKE '%(NCR)%' THEN 'NATIONAL CAPITAL REGION'
      WHEN TRIM(region) LIKE '%(NIR)%' THEN 'NEGROS ISLAND REGION'
      WHEN TRIM(region) LIKE '%(CALABARZON)%' OR TRIM(region) = 'REGION IVA CALABARZON' THEN 'CALABARZON'
      WHEN TRIM(region) LIKE '%(ILOCOS REGION)%' THEN 'ILOCOS REGION'
      WHEN TRIM(region) LIKE '%(CAGAYAN VALLEY)%' THEN 'CAGAYAN VALLEY'
      WHEN TRIM(region) LIKE '%(CENTRAL LUZON)%' THEN 'CENTRAL LUZON'
      WHEN TRIM(region) LIKE '%(ZAMBOANGA PENINSULA)%' THEN 'ZAMBOANGA PENINSULA'
      WHEN TRIM(region) LIKE '%(BICOL REGION)%' THEN 'BICOL REGION'
      WHEN TRIM(region) LIKE '%(WESTERN VISAYAS)%' THEN 'WESTERN VISAYAS'
      WHEN TRIM(region) LIKE '%(CENTRAL VISAYAS)%' THEN 'CENTRAL VISAYAS'
      WHEN TRIM(region) LIKE '%(EASTERN VISAYAS)%' THEN 'EASTERN VISAYAS'
      WHEN TRIM(region) LIKE '%(NORTHERN MINDANAO)%' THEN 'NORTHERN MINDANAO'
      WHEN TRIM(region) LIKE '%(DAVAO REGION)%' THEN 'DAVAO REGION'
      WHEN TRIM(region) LIKE '%(SOCCSKSARGEN)%' THEN 'SOCCSKSARGEN'
      WHEN TRIM(region) LIKE '%(CARAGA)%' THEN 'CARAGA'
      WHEN TRIM(region) = 'MIMAROPA REGION' THEN 'MIMAROPA REGION'
    END AS region_name,

    MIN(TRY_CAST(year AS INT)) AS valid_from_year,
    MAX(TRY_CAST(year AS INT)) AS valid_to_year,
    'region' AS geographic_level

  FROM pondolens.dev_bronze.unemployment_rate
  WHERE LOWER(TRIM(geographic_level)) = 'region'
    AND TRY_CAST(year AS INT) BETWEEN 2021 AND 2025
  GROUP BY TRIM(region)
) AS source
ON target.source_region = source.source_region
AND target.geographic_level = source.geographic_level

WHEN MATCHED THEN UPDATE SET
  target.country = source.country,
  target.region_code = source.region_code,
  target.region_name = source.region_name,
  target.valid_from_year = source.valid_from_year,
  target.valid_to_year = source.valid_to_year,
  target.source_code = source.source_code

WHEN NOT MATCHED THEN INSERT (
  country, source_region, region_code, region_name,
  valid_from_year, valid_to_year, source_code, geographic_level
)
VALUES (
  source.country, source.source_region, source.region_code, source.region_name,
  source.valid_from_year, source.valid_to_year, source.source_code, source.geographic_level
)


SELECT *
FROM pondolens.dev_silver.shared_region_crosswalk
WHERE geographic_level = 'region';



-------------------------------------------------------------------------



SELECT DISTINCT
  TRIM(region) AS source_region,
  CASE
    WHEN TRIM(region) LIKE '%(BARMM/ARMM)%' THEN 'BARMM/ARMM'
    WHEN TRIM(region) LIKE '%(BARMM)%' THEN 'BARMM'
    WHEN TRIM(region) LIKE '%(CAR)%' THEN 'CAR'
    WHEN TRIM(region) LIKE '%(NCR)%' THEN 'NCR'
    WHEN TRIM(region) LIKE '%(NIR)%' THEN 'NIR'
    WHEN TRIM(region) LIKE '%(CALABARZON)%'
      OR TRIM(region) = 'REGION IVA CALABARZON' THEN 'IVA'
    WHEN TRIM(region) LIKE '%(ILOCOS REGION)%' THEN 'I'
    WHEN TRIM(region) LIKE '%(CAGAYAN VALLEY)%' THEN 'II'
    WHEN TRIM(region) LIKE '%(CENTRAL LUZON)%' THEN 'III'
    WHEN TRIM(region) LIKE '%(ZAMBOANGA PENINSULA)%' THEN 'IX'
    WHEN TRIM(region) LIKE '%(BICOL REGION)%' THEN 'V'
    WHEN TRIM(region) LIKE '%(WESTERN VISAYAS)%' THEN 'VI'
    WHEN TRIM(region) LIKE '%(CENTRAL VISAYAS)%' THEN 'VII'
    WHEN TRIM(region) LIKE '%(EASTERN VISAYAS)%' THEN 'VIII'
    WHEN TRIM(region) LIKE '%(NORTHERN MINDANAO)%' THEN 'X'
    WHEN TRIM(region) LIKE '%(DAVAO REGION)%' THEN 'XI'
    WHEN TRIM(region) LIKE '%(SOCCSKSARGEN)%' THEN 'XII'
    WHEN TRIM(region) LIKE '%(CARAGA)%' THEN 'XIII'
    WHEN TRIM(region) = 'MIMAROPA REGION' THEN 'IVB'
  END AS region_code
FROM pondolens.dev_bronze.unemployment_rate
WHERE LOWER(TRIM(geographic_level)) = 'region'
  AND TRY_CAST(year AS INT) BETWEEN 2021 AND 2025;


  SELECT
  geographic_level,
  year,
  region
FROM pondolens.dev_bronze.unemployment_rate
LIMIT 20;


------------------------------------------------------------------------------